# API quickstart — Multicellular Transformer

Minimal end-to-end path using the `cell_cell_interactions.api` layer. Old imports still work; use this notebook if you're starting fresh.

**Cells are illustrative (unexecuted).** Run against a local checkpoint and adjust paths.

## 1. Load a checkpoint + its paired index files

In [ ]:
from cell_cell_interactions.api import load_model

model = load_model(
    checkpoint_path="checkpoints/multicellular_transformer.ckpt",
    index_dir="tutorials/data",  # gene2idx.json / cell_type_input_order.json / tissue2idx.json
).to("cuda")

print(model.config)
print("vocab size:", len(model.index.gene2idx))

## 2. Zero-shot gene embeddings (no data required)

Replaces the raw `state_dict['model.gene_id_classifier.decoder.weight']` slice used in the paper tutorials.

In [ ]:
gene_embeds = model.gene_embeddings(["EGFR", "TP53", "MYC"])
print({g: v.shape for g, v in gene_embeds.items()})

all_embeds = model.gene_embeddings()  # every gene in the vocabulary
print("n genes:", len(all_embeds))

## 3. Context-aware embeddings on your own AnnData

In [ ]:
from cell_cell_interactions.api import AdataInferencePreset, embed_adata

preset = AdataInferencePreset(
    adata_file_path="/path/to/your.h5ad",
    gene2idx=model.index.gene2idx,
    label_column="cell_type",
    sample_column="sample",
    tissue_column="tissue",
)

results = embed_adata(model, preset, max_batches=5)
print("batches:", len(results))
print("first batch hidden state:", results[0]["last_hidden_state"].shape)

## 4. Per-cell gene-symbol map (no more hand-written demux)

Replaces the per-notebook `get_gene_embeddings(...)` helper.

In [ ]:
from cell_cell_interactions.api import get_context_aware_gene_embeddings, embed_dataloader

dm = preset.build()
results = embed_dataloader(model, dm.val_dataloader(), return_batches=True, max_batches=1)
first = results[0]

# Recover per-cell {gene_symbol: vector} dicts
output_ns = type("O", (), {"last_hidden_state": __import__("torch").from_numpy(first["last_hidden_state"])})()
per_cell = get_context_aware_gene_embeddings(model, output_ns, first["batch"])
print("n cells in first batch:", len(per_cell))
print("genes seen in cell 0:", list(per_cell[0])[:10])

## 5. Downstream analysis

Same tools as the paper tutorials, imported from one place.

In [ ]:
from cell_cell_interactions.api import (
    GeneEmbedding,
    calculate_condition_specificity,
    plot_gene_set,
    plot_similarity_across_conditions_heatmap,
)

# Zero-shot gene co-regulation and metagene clustering
embed = GeneEmbedding(all_embeds)
gdata = embed.get_adata(resolution=40)
metagenes = embed.get_metagenes(gdata)
print("metagene clusters:", len(metagenes))

# Condition-specific genes example: load a precomputed neighbor table
import pandas as pd
neighbors_df = pd.read_parquet("tutorials/data/immune_across_organs_neighbors_sample.parquet")
condition_specific = calculate_condition_specificity(
    df=neighbors_df,
    condition="tissue",
    min_n_conditions=2,
    min_n_samples=3,
    k_neighbors_within_condition=10,
    k_neighbors_across_conditions=10,
)
print(condition_specific.head())

## 6. Sample-level and DepMap KO workflows

In [ ]:
from cell_cell_interactions.utils.sample_embeddings.api import (
    collect_sample_embeddings,
    collect_sample_neighbors,
)
from cell_cell_interactions.utils.depmap.api import run_gene_ko_effect, DEPMAP_DISEASE_MAP

# Example: sample embeddings over your dataloader
# stacked, meta = collect_sample_embeddings(model, dm.train_dataloader(), "outputs/")

# Example: in-silico DepMap KO on a datamodule configured with agg_cell_type_gex=True
# expr, rel_expr = run_gene_ko_effect(
#     model=model,
#     dataloader=dm.train_dataloader(),
#     tumor_cell_type="Malignant cells",
#     genes_to_knock_out=["EGFR", "KRAS"],
#     name2idx={"EGFR": model.index.gene2idx["EGFR"], "KRAS": model.index.gene2idx["KRAS"]},
#     output_dir="outputs/ko",
#     max_steps=10,
# )